In [3]:
## 일단 다른 작업 하지 말고 (질병 통일) 

df = pd.read_csv('00NNEEWW_KNHANES_lipid_grouped.csv')

analysis_df = df.copy()

analysis_df = analysis_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# =========================================================
# 2. ASCVD 및 Diabetes 변수 생성
# =========================================================

# ASCVD:
# DI5_dg 또는 DI6_dg가 1인 경우
analysis_df["ASCVD"] = (
    (analysis_df["DI5_dg"] == 1) |
    (analysis_df["DI6_dg"] == 1)
).astype(int)


# Diabetes mellitus(당뇨병):
# DE1_dg, HE_DM_HbA1c 중 하나라도 1인 경우
analysis_df["Diabetes"] = (
    (analysis_df["DE1_dg"] == 1) |
    (analysis_df["HE_DM_HbA1c"] == 3)
).astype(int)


# =========================================================
# 3. Disease 문자열 변수 생성
# =========================================================
conditions = [
    (
        (analysis_df["ASCVD"] == 1) &
        (analysis_df["Diabetes"] == 1)
    ),
    (
        (analysis_df["ASCVD"] == 1) &
        (analysis_df["Diabetes"] == 0)
    ),
    (
        (analysis_df["ASCVD"] == 0) &
        (analysis_df["Diabetes"] == 1)
    )
]

choices = [
    "ASCVD + Diabetes",
    "ASCVD",
    "Diabetes"
]

analysis_df["Disease"] = np.select(
    conditions,
    choices,
    default="Others"
)

print("Disease 분포:")
print(analysis_df["Disease"].value_counts(dropna=False))

Disease 분포:
Disease
Others              6635
Diabetes            2521
ASCVD                167
ASCVD + Diabetes     141
Name: count, dtype: int64


In [8]:
analysis_df.shape

(9464, 42)

In [11]:
# ==========================================================
# ASCVD 환자군
# ==========================================================

ascvd_df = analysis_df[
    (analysis_df["Diabetes"] == 1) &
    (analysis_df["ASCVD"] == 1) &
    (analysis_df["HE_LDL_drct"].notna())&
    (analysis_df["Patient_Group"].isin([
        "Control",
        "Inadequately Treated"
    ]))
].copy()

total_n = len(ascvd_df)

achieved_n = (
    ascvd_df["Patient_Group"]
    == "Control"
).sum()

unachieved_n = (
    ascvd_df["Patient_Group"]
    == "Inadequately Treated"
).sum()

achieved_rate = achieved_n / total_n * 100
unachieved_rate = unachieved_n / total_n * 100

print("=" * 80)
print("ASCVD and Diabetes군 LDL-C 목표 달성 여부")
print("=" * 80)

print(f"분석 대상자 수: {total_n:,}")
print(
    f"목표 달성(Control): "
    f"{achieved_n:,}명 "
    f"({achieved_rate:.2f}%)"
)
print(
    f"목표 미달성(Inadequately Treated): "
    f"{unachieved_n:,}명 "
    f"({unachieved_rate:.2f}%)"
)

# ==========================================================
# 목표 미달성자의 LDL-C
# ==========================================================

ascvd_unachieved_df = ascvd_df[
    (ascvd_df["Patient_Group"] == "Inadequately Treated") &
    (ascvd_df["HE_LDL_drct"].notna())
]

print("\n미달성 환자 LDL-C")
print(f"표본수 (N)         : {len(ascvd_unachieved_df):,}")
print(
    f"LDL-C 평균         : "
    f"{ascvd_unachieved_df['HE_LDL_drct'].mean():.2f} mg/dL"
)
print(
    f"LDL-C 표준편차(SD) : "
    f"{ascvd_unachieved_df['HE_LDL_drct'].std():.2f} mg/dL"
)

ASCVD and Diabetes군 LDL-C 목표 달성 여부
분석 대상자 수: 118
목표 달성(Control): 18명 (15.25%)
목표 미달성(Inadequately Treated): 100명 (84.75%)

미달성 환자 LDL-C
표본수 (N)         : 100
LDL-C 평균         : 84.08 mg/dL
LDL-C 표준편차(SD) : 26.66 mg/dL


In [9]:
# ==========================================================
# ASCVD 환자군 LDL-C <55 mg/dL 달성 여부
# ==========================================================

ascvd_df = analysis_df[
    (analysis_df["ASCVD"] == 1) &
    (analysis_df["HE_LDL_drct"].notna())
].copy()

total_n = len(ascvd_df)

achieved_df = ascvd_df[
    ascvd_df["HE_LDL_drct"] < 55
]

unachieved_df = ascvd_df[
    ascvd_df["HE_LDL_drct"] >= 55
]

achieved_n = len(achieved_df)
unachieved_n = len(unachieved_df)

print("=" * 80)
print("ASCVD군 LDL-C <55 mg/dL 미달성")
print("=" * 80)

print(f"분석 대상자 수: {total_n:,}")
print(
    f"목표 달성: {achieved_n:,}명 "
    f"({achieved_n / total_n * 100:.2f}%)"
)
print(
    f"목표 미달성: {unachieved_n:,}명 "
    f"({unachieved_n / total_n * 100:.2f}%)"
)

print("\n미달성 환자 LDL-C")
print(f"표본수 (N)         : {unachieved_n:,}")
print(
    f"LDL-C 평균         : "
    f"{unachieved_df['HE_LDL_drct'].mean():.2f} mg/dL"
)
print(
    f"LDL-C 표준편차(SD) : "
    f"{unachieved_df['HE_LDL_drct'].std():.2f} mg/dL"
)

ASCVD군 LDL-C <55 mg/dL 미달성
분석 대상자 수: 296
목표 달성: 41명 (13.85%)
목표 미달성: 255명 (86.15%)

미달성 환자 LDL-C
표본수 (N)         : 255
LDL-C 평균         : 94.39 mg/dL
LDL-C 표준편차(SD) : 33.15 mg/dL
